In [1]:
import xarray as xr
import s3fs

In [2]:
endpoint_url = "https://objectstore.eodc.eu:2222"
bucket = "e05ab01a9d56408d82ac32d69a5aae2a:sample-data"
prefix = "tutorial_data/cpm_v253"

path = "S2B_MSIL1C_20250113T103309_N0511_R108_T32TLQ_20250113T122458.zarr"

In [3]:
# Create the S3FileSystem with a custom endpoint
fs = s3fs.S3FileSystem(
    anon=True,
    client_kwargs={
        "endpoint_url": endpoint_url
    }
)

# unregister handler to make boto3 work with CEPH
handlers = fs.s3.meta.events._emitter._handlers
handlers_to_unregister = handlers.prefix_search("before-parameter-build.s3")
handler_to_unregister = handlers_to_unregister[0]
fs.s3.meta.events._emitter.unregister(
    "before-parameter-build.s3", handler_to_unregister
)

# List all objects in the bucket
#files = fs.ls(f"{bucket}/{prefix}/")  # Set detail=True to get metadata
#files

In [4]:
mapper = fs.get_mapper(root=f"s3://{bucket}/{prefix}/{path}" + "/measurements/reflectance/r10m")

import json
import io
zarray = json.load(io.BytesIO(mapper["b02/.zarray"]))

zarray

{'chunks': [1830, 1830],
 'compressor': {'blocksize': 0,
  'clevel': 3,
  'cname': 'zstd',
  'id': 'blosc',
  'shuffle': 2},
 'dtype': '<u2',
 'fill_value': 0,
 'filters': None,
 'order': 'C',
 'shape': [10980, 10980],
 'zarr_format': 2}

In [5]:
dataset = xr.open_dataset(mapper, engine="zarr", chunks="auto")
dataset.b02

C:\Users\norma\miniforge3\envs\eopf-xr\Lib\site-packages\xarray\backends\plugins.py:110: RuntimeWarning: Engine 'eopf-zarr' loading failed:
invalid syntax (constants.py, line 29)
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
C:\Users\norma\AppData\Local\Temp\ipykernel_7116\4075965386.py:1: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  dataset = xr.open_dataset(mapper, engine="zarr", chunks="auto")


<xarray.DataArray 'b02' (y: 10980, x: 10980)> Size: 964MB
dask.array<open_dataset-b02, shape=(10980, 10980), dtype=float64, chunksize=(3660, 3660), chunktype=numpy.ndarray>
Coordinates:
  * x        (x) int64 88kB 300005 300015 300025 300035 ... 409775 409785 409795
  * y        (y) int64 88kB 5000035 5000025 5000015 ... 4890265 4890255 4890245
Attributes:
    _eopf_attrs:     {'add_offset': -0.1, 'coordinates': ['x', 'y'], 'dimensi...
    dtype:           <u2
    fill_value:      0
    long_name:       TOA reflectance from MSI acquisition at spectral band b0...
    proj:bbox:       [300000.0, 4890240.0, 409800.0, 5000040.0]
    proj:epsg:       32632
    proj:shape:      [10980, 10980]
    proj:transform:  [10.0, 0.0, 300000.0, 0.0, -10.0, 5000040.0, 0.0, 0.0, 1.0]
    proj:wkt2:       PROJCS["WGS 84 / UTM zone 32N",GEOGCS["WGS 84",DATUM["WG...
    units:           digital_counts
    valid_max:       65535
    valid_min:       1

---
# Sentinel 2 L1C

In [6]:
path = (
    "https://objectstore.eodc.eu:2222/e05ab01a9d56408d82ac32d69a5aae2a:sample-data/tutorial_data/"
    "cpm_v253/S2B_MSIL1C_20250113T103309_N0511_R108_T32TLQ_20250113T122458.zarr"
)

In [7]:
ds = xr.open_dataset(path, engine="eopf-zarr", op_mode="native")

ValueError: unrecognized engine 'eopf-zarr' must be one of your download engines: ['scipy', 'store', 'zarr']. To install additional dependencies, see:
https://docs.xarray.dev/en/stable/user-guide/io.html 
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html

In [ ]:
from xarray_eopf.util.spatial import get_spatial_vars, get_ref_var_name, rescale_spatial_vars

In [ ]:
%%time
spatial_vars = get_spatial_vars(ds.data_vars)

In [ ]:
list(spatial_vars.keys())

In [ ]:
%%time
ref_var_name = get_ref_var_name(ds.data_vars)

In [ ]:
ref_var_name

In [ ]:
%%time
rescaled_spatial_vars = rescale_spatial_vars(spatial_vars)

In [ ]:
list(rescaled_spatial_vars.keys())